# Init

In [8]:
init_training_sentences = ["the cat sat on the mat", "the dog sat on the chair", "the guy sat on the sofa"]
init_vocab_size = len(set([word for sentence in init_training_sentences for word in sentence.split()])) # unique words in training sentences
init_embedding_dimension = 8
init_attention_dimension = 4 # usually better if it's half of embedding dimension
init_input = "the cat sat on the"
init_input_length = len(init_input.split())

## Tokenization

In [9]:
# Get list of unique words

print("Tokenization\n")

unique_words = sorted(set([word for sentence in init_training_sentences for word in sentence.split()]))

print("unique_words: ", unique_words)

word_to_id = {
    word: i for i, word in enumerate(unique_words)
}

print("word_to_id: ", word_to_id)

id_to_word = {
    i: word for word, i in word_to_id.items()
}

# get input and encode it
input_encoded = [word_to_id[input] for input in init_input.split()]

print("\ninput_encoded: ", input_encoded)


Tokenization

unique_words:  ['cat', 'chair', 'dog', 'guy', 'mat', 'on', 'sat', 'sofa', 'the']
word_to_id:  {'cat': 0, 'chair': 1, 'dog': 2, 'guy': 3, 'mat': 4, 'on': 5, 'sat': 6, 'sofa': 7, 'the': 8}

input_encoded:  [8, 0, 6, 5, 8]


## Embedding

In [4]:
import numpy as np

print("Embedding\n")
embedding_table = np.random.randn(init_vocab_size, init_embedding_dimension)

print("embedding_table.shape: \n", embedding_table.shape)
print("input_encoded: \n", input_encoded)

embeddings = embedding_table[input_encoded]

print("\nembeddings.shape: ", embeddings.shape)

Embedding

embedding_table.shape: 
 (9, 8)
input_encoded: 
 [8, 0, 6, 5, 8]

embeddings.shape:  (5, 8)


## Attention

In [5]:
# Create attention layer

# Who am I?
W_K = np.random.randn(init_embedding_dimension, init_attention_dimension)
# What am I looking for?
W_Q = np.random.randn(init_embedding_dimension, init_attention_dimension)
# What information do I pass forward?
W_V = np.random.randn(init_embedding_dimension, init_attention_dimension)

print("embeddings.shape: ", embeddings.shape)
print("W_K.shape: ", W_K.shape)

# Create Keys
K = embeddings.dot(W_K)
# Create Queries
Q = embeddings.dot(W_Q)
# Create Values
V = embeddings.dot(W_V)

print("K.shape: embeddings @ W_K: ", K.shape)

# Compare every query against every key
scores = Q.dot(K.T)
print("Q: ", Q.shape)
print("Q @ K: ", scores.shape)

# Normalize scores
scores = scores / np.sqrt(init_attention_dimension)


embeddings.shape:  (5, 8)
W_K.shape:  (8, 4)
K.shape: embeddings @ W_K:  (5, 4)
Q:  (5, 4)
Q @ K:  (5, 5)


## Softmax

In [7]:
# Convert scores into probabilities
print("scores: \n\n", scores)

# upper triangle = future tokens
mask = np.triu(np.ones(scores.shape), k=1)

# Prevent future token attention
scores = np.where(mask == 1, -np.inf, scores)

print("causal masking:\n\n", scores)

scores = scores - np.max(scores, axis=-1, keepdims=True)

print("scores - max: \n\n", scores)

exp_scores = np.exp(scores)

print("exp_scores: \n\n", exp_scores)

attention_weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)

print("attention_weights: \n\n", attention_weights)

attention_output = attention_weights.dot(V)

# final prediction matrix
# lm_head
vocabulary_scores = np.random.randn(init_attention_dimension, len(unique_words))

# raw word scores
# logits
vocabulary_logits = attention_output @ vocabulary_scores

# use last token's vocabulary_logits for next-word prediction
last_vocabulary_logits = vocabulary_logits[-1]

# softmax over vocabulary
exponentiated_vocabulary_logits = np.exp(last_vocabulary_logits - np.max(last_vocabulary_logits))
probabilities = exponentiated_vocabulary_logits / np.sum(exponentiated_vocabulary_logits)

top_indices = np.argsort(probabilities)[-5:][::-1]

print("top_indices: ", top_indices)

for idx in top_indices:
    bar = "█" * int(probabilities[idx] * 50)
    print(f"{unique_words[idx]:10s} {probabilities[idx]:.3f} {bar}")

scores: 

 [[  0.                 -inf         -inf         -inf         -inf]
 [  0.          -5.8086356          -inf         -inf         -inf]
 [  0.          -0.67012357  -0.47686365         -inf         -inf]
 [ -2.92354211   0.          -3.50605268  -8.84409996         -inf]
 [  0.         -12.06293276 -12.02145842  -5.31637674   0.        ]]
causal masking:

 [[  0.                 -inf         -inf         -inf         -inf]
 [  0.          -5.8086356          -inf         -inf         -inf]
 [  0.          -0.67012357  -0.47686365         -inf         -inf]
 [ -2.92354211   0.          -3.50605268  -8.84409996         -inf]
 [  0.         -12.06293276 -12.02145842  -5.31637674   0.        ]]
scores - max: 

 [[  0.                 -inf         -inf         -inf         -inf]
 [  0.          -5.8086356          -inf         -inf         -inf]
 [  0.          -0.67012357  -0.47686365         -inf         -inf]
 [ -2.92354211   0.          -3.50605268  -8.84409996         -inf]
